# Stock Data Quality Exploration

## Objective

This notebook investigates whether the raw daily OHLCV dataset is
suitable for factor research and backtesting.

## 0. Load the dataset

In [39]:
import pandas as pd
stock = pd.read_csv("../data/raw/dirty_stock_daily_ohlcv.csv")
stock.head()

,ticker,date,open,high,low,close,volume
0,NVDA,2025/2/20,911.45,929.59,859.46,871.51,18492144.0
1,NVDA,2025/2/6,880.17,908.99,873.47,900.50,65147551.0
2,BABA,2025/3/3,79.21,79.67,78.40,78.69,35511162.0
3,NVDA,2025/2/21,873.90,899.37,868.46,891.95,23404601.0
4,MSFT,2025/2/12,451.12,458.86,450.63,458.11,31058226.0


## 1. How many rows are there?

In [40]:
row_count = len(stock)
print("Number of rows:",row_count)

Number of rows: 517


### Interpretation
The dataset contains 517 rows. Each row represents one stock on one trading date.

## 2. How many stocks are there?

In [41]:
stock_count = stock["ticker"].nunique()
print("Numbers of stocks:",stock_count)

Numbers of stocks: 6


### Interpretation
The dataset contains 6 unique stocks, identified by the `ticker` column.

## 3. Are the required columns present? Is there any unexpected column?

In [42]:
expected_columns = {"ticker","date","open","close","high","low","volume"}
actual_columns = set(stock.columns)
unexpected_columns = actual_columns-expected_columns
missing_columns = expected_columns-actual_columns
print("missing columns:",missing_columns)
print("unexpected columns:",unexpected_columns)

missing columns: set()
unexpected columns: set()


### Interpretation
The dataset doesn't miss any expected column, and it doesn't have unexpected columns.

## 4. Are the date types appropriate?

In [43]:
print(stock.dtypes)

ticker     object
date       object
open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object


## 5. Are there missing values?

In [44]:
missing_summary = pd.DataFrame({"missing_count":stock.isna().sum(),"missing_ratio":stock.isna().mean()})
print(missing_summary)

        missing_count  missing_ratio
ticker              0       0.000000
date                0       0.000000
open                3       0.005803
high                2       0.003868
low                 2       0.003868
close               3       0.005803
volume              2       0.003868


### Interpretation
The table reports the number and proportion of missing values in each column.

## 6. Are there repeated rows?

In [ ]:
duplicate_mask = stock.duplicated(subset=["ticker", "date"],keep=False,)
duplicate_rows = stock[duplicate_mask]
duplicate_rows

,ticker,date,open,high,low,close,volume
35,BABA,2025/4/18,78.12,78.57,77.59,77.97,22825520.0
46,AAPL,2025/4/7,193.09,195.97,191.51,191.56,50212420.0
76,MSFT,2025/2/3,427.24,433.74,424.80,428.00,41465830.0
81,JPM,2025/3/21,0.01,0.02,0.01,0.02,999999999.0
93,AAPL,2025/1/17,192.15,194.21,191.21,193.93,29300785.0
132,NVDA,2025/3/11,928.55,937.86,919.02,932.47,4079416.0
244,TSLA,2025/2/14,999.99,1005.00,995.00,1001.20,123456789.0
303,NVDA,2025/3/11,928.55,937.86,919.02,932.47,4079416.0
318,TSLA,2025/2/14,328.25,339.29,325.81,338.75,70441604.0
322,AAPL,2025/4/7,185.12,187.90,184.50,186.33,100.0


### Interpretation
Each stock should have at most one daily record for each trading date.

We need to inspect these duplicate rows before we remove them

## 7. Are there negative or zero prices?

In [46]:
negative_rows = stock[(stock["high"]<=0)|(stock["low"]<=0)|(stock["open"]<=0)|(stock["close"]<=0)]
negative_rows

,ticker,date,open,high,low,close,volume
25,TSLA,2025/1/31,309.09,310.00,-3.50,303.55,76726941.0
256,AAPL,2025/3/17,-12.34,-11.80,-13.10,-12.90,65463826.0
292,BABA,2025/3/19,0.00,0.00,0.00,0.00,28290566.0
509,MSFT,2025/4/10,423.41,436.66,420.12,-420.00,64736692.0


### Interpretation
Each price has to be positive.

We need to inspect these rows which have negative values before we remove them.

## 8. Are there negative volumes?

In [47]:
negative_volumes = stock[stock["volume"]<0]
negative_volumes

,ticker,date,open,high,low,close,volume
98,AAPL,2025/4/22,189.71,192.5,188.1,191.76,-500000.0


### Interpretation
Each volume shouldn't be negative.

We need to inspect these rows which have negative volume before we remove them.

## 9. Are there obviously unreasonable OHLC relations?

In [48]:
weird_rows = stock[(stock["close"]>stock["high"]) | (stock["close"]<stock["low"]) | (stock["open"]>stock["high"]) | (stock["open"]<stock["low"])]
weird_rows

,ticker,date,open,high,low,close,volume
60,JPM,2025/2/11,202.26,1.00,210.00,206.64,59396522.0
509,MSFT,2025/4/10,423.41,436.66,420.12,-420.00,64736692.0


### Interpretation
Close and open price have to be smaller than highest price and bigger than lowest price.

We need to inspect these rows which have unreasonable OHLC relations before we remove them.

## 10. Findings and next steps

### Findings

- The dataset contains **517 rows** and **6 unique stocks**.
- All required columns are present.
- Several OHLC and volume values are missing.
- Some `ticker`–`date` combinations appear more than once.
- Some rows contain invalid non-positive prices.
- At least one row contains a negative trading volume.
- Some rows violate OHLC consistency rules.

### Cleaning decisions

- Preserve the original raw dataset without modification.
- Parse and validate dates before time-series analysis.
- Review duplicate `ticker`–`date` records and distinguish exact duplicates
  from conflicting records.
- Remove or correct rows with non-positive prices.
- Remove or correct rows with negative trading volume.
- Remove or correct rows with invalid OHLC relationships.
- Save the cleaned dataset separately from the raw dataset.

### Limitations

These checks are necessary but not sufficient to prove that the dataset is
ready for backtesting. A second-stage analysis should inspect extreme returns,
zero-volume records, missing trading days, corporate actions, adjusted prices,
and survivorship bias.